In [30]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [31]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = make_moons(
    n_samples=1200,
    noise=0.25,
    random_state=42
)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val)

In [32]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [33]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),

    tf.keras.layers.Dense(32, use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Dense(16, use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=['accuracy'])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 32)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 16)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 785 (3.07 KB)

 Trainable params: 689 (2.69 KB)

 Non-trainable params: 96 (384.00 B)

In [34]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    min_delta=0.0001,
    restore_best_weights=True
)

In [35]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7865 - loss: 0.4926 - val_accuracy: 0.8021 - val_loss: 0.5767
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 951us/step - accuracy: 0.8451 - loss: 0.3703 - val_accuracy: 0.8646 - val_loss: 0.5001
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 885us/step - accuracy: 0.8529 - loss: 0.3393 - val_accuracy: 0.8750 - val_loss: 0.4562
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 872us/step - accuracy: 0.8594 - loss: 0.3244 - val_accuracy: 0.8698 - val_loss: 0.4281
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 822us/step - accuracy: 0.8633 - loss: 0.3125 - val_accuracy: 0.8750 - val_loss: 0.4070
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 854us/step - accuracy: 0.8750 - loss: 0.3025 - val_accuracy: 0.8698 - val_loss: 0.3875
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 832us/step - accuracy: 0.8776 - loss: 0.2938 - val_accuracy: 0.8750 - val_loss: 0.3674
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - accuracy: 0.8789 - loss: 0.2868 - val_accura

In [36]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\nTest Loss : ", round(test_loss, 4))
print("Test Accuracy : ", round(test_accuracy, 4))


Test Loss :  0.1612
Test Accuracy :  0.9417


In [37]:
from sklearn.metrics import classification_report
probabilities = model.predict(X_test, verbose=0)
predictions = (probabilities >= 0.5).astype(int).ravel()
print("\nClassification Report")
print(classification_report(y_test, predictions))


Classification Report
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       120
           1       0.94      0.94      0.94       120

    accuracy                           0.94       240
   macro avg       0.94      0.94      0.94       240
weighted avg       0.94      0.94      0.94       240

